In [1]:
# Minimal reproducibility bootstrap for this notebook
import warnings
import random
import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Bootstrap complete. Seed={SEED}")


Bootstrap complete. Seed=42


# Competitive Intelligence Crew

## What We're Building

A four-agent competitive intelligence workflow that tracks competitor features, pricing, launches, and outputs one weekly memo.

| Agent | Specialization |
|-------|----------------|
| Competitor Tracker | Detect and categorize competitor moves by strategic significance |
| Launch Summarizer | Interpret launch announcements and assess threat urgency |
| Feature/Pricing Analyst | Build feature-price comparison and win/loss matrix |
| Memo Writer | Synthesize all findings into an executive weekly memo |

## Agent Specialization Design

- Tracker focuses on signal detection, not recommendations.
- Launch Summarizer focuses on launch impact interpretation.
- Feature/Pricing Analyst focuses on competitive mechanics (price and capability).
- Memo Writer focuses on executive communication and action prioritization.

This keeps each role narrow and improves consistency of the final memo.

## Stack
- CrewAI - multi-agent orchestration
- Ollama - local LLM endpoint


In [2]:
CREWAI_AVAILABLE = True

try:
    from crewai import Agent, Task, Crew, Process, LLM
except Exception:
    CREWAI_AVAILABLE = False
    Agent = Task = Crew = Process = LLM = None
    print("CrewAI is not installed. Install with: pip install crewai")

llm = None
if CREWAI_AVAILABLE:
    llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
    print(f"LLM ready: {llm.model}")
else:
    print("Running in demo mode without CrewAI runtime.")


CrewAI is not installed. Install with: pip install crewai
Running in demo mode without CrewAI runtime.


## Step 2 — Competitor Data

In [3]:
our_company = "DataSync Pro — B2B data integration platform"

# Simulated competitor intelligence (in production, this would come from web scraping)
competitor_updates = """
COMPETITOR UPDATE FEED (past 2 weeks):

--- COMPETITOR 1: Fivetran ---
- Announced SOC 2 Type II certification for all connectors
- Launched 15 new connectors including Notion, Linear, and Airtable
- Raised pricing: Starter plan now $500/mo (was $400/mo)
- New feature: "QuickStart" templates for common data pipelines
- Blog post: "Why ETL is dead — long live ELT" (2,500 shares)
- Hired VP of Enterprise Sales from Snowflake

--- COMPETITOR 2: Airbyte ---
- Released Airbyte Cloud 2.0 with AI-powered schema mapping
- Open-source repo hit 15,000 GitHub stars
- Announced $200M Series C at $1.5B valuation
- New feature: "Connector Builder" — lets users create custom connectors in 10 min
- Dropped prices 20% across all tiers
- Launched developer certification program

--- COMPETITOR 3: Stitch (by Talend) ---
- Quiet period — no major product announcements
- Reduced marketing spend (fewer blog posts, social activity down 40%)
- Lost 2 senior engineers to Airbyte (LinkedIn confirms)
- Still advertising at old pricing ($100/mo starter)
- Talend parent company reported flat Q3 revenue
"""

print("Competitor update feed loaded (3 competitors)")

Competitor update feed loaded (3 competitors)


## Step 3 — Create Agents

In [4]:
if CREWAI_AVAILABLE:
    tracker = Agent(
        role="Competitor Tracker",
        goal="Monitor and categorize competitor activities by strategic importance",
        backstory=(
            "You are a competitive intelligence specialist who watches pricing, hiring,"
            " launch, and messaging signals to detect strategic movement."
        ),
        llm=llm,
        verbose=True,
    )

    launch_summarizer = Agent(
        role="Launch Summarizer",
        goal="Summarize competitor launches and assess likely market impact",
        backstory=(
            "You evaluate product launches for novelty, defensibility, and customer pull."
        ),
        llm=llm,
        verbose=True,
    )

    feature_pricing_analyst = Agent(
        role="Feature/Pricing Analyst",
        goal="Extract pricing tiers and feature differentials into a win/loss matrix",
        backstory=(
            "You support product and sales teams with structured competitor comparisons"
            " focused on capability and price shifts."
        ),
        llm=llm,
        verbose=True,
    )

    memo_writer = Agent(
        role="Weekly Memo Writer",
        goal="Produce a concise weekly memo with clear priorities and recommended actions",
        backstory=(
            "You write executive briefs that are short, evidence-based, and actionable."
        ),
        llm=llm,
        verbose=True,
    )
else:
    tracker = {"role": "Competitor Tracker"}
    launch_summarizer = {"role": "Launch Summarizer"}
    feature_pricing_analyst = {"role": "Feature/Pricing Analyst"}
    memo_writer = {"role": "Weekly Memo Writer"}

print("4 agents created: Competitor Tracker, Launch Summarizer, Feature/Pricing Analyst, Memo Writer")


4 agents created: Competitor Tracker, Launch Summarizer, Feature/Pricing Analyst, Memo Writer


## Step 4 — Create Tasks

In [5]:
if CREWAI_AVAILABLE:
    track_task = Task(
        description=f"""Analyze this competitor update feed and categorize each activity:

Our company: {our_company}

{competitor_updates}

For each competitor, classify moves as high, medium, low, or opportunity,
and flag momentum signals.""",
        expected_output="Categorized competitor activity feed with threat levels.",
        agent=tracker,
    )

    launch_task = Task(
        description="""For each launch/update identified, provide:
1. What changed
2. Why it matters
3. Recommended response
4. Threat score (1-10)
5. Response urgency""",
        expected_output="Launch impact assessments with threat scores and response recommendations.",
        agent=launch_summarizer,
        context=[track_task],
    )

    feature_task = Task(
        description=f"""Create feature and pricing comparison for:
Our product: {our_company}

Include tier pricing, key feature differences, and win/loss/tie markers.""",
        expected_output="Feature-pricing comparison matrix with win/loss analysis.",
        agent=feature_pricing_analyst,
        context=[track_task],
    )

    memo_task = Task(
        description="""Write the weekly competitive memo in this structure:
1. BLUF (3 lines)
2. Key moves this week
3. Feature/pricing watch
4. Competitor health check
5. Recommended actions

Limit to 500 words.""",
        expected_output="Weekly competitive intelligence memo.",
        agent=memo_writer,
        context=[track_task, launch_task, feature_task],
        markdown=True,
    )
else:
    track_task = {"name": "Tracking task"}
    launch_task = {"name": "Launch summarization task"}
    feature_task = {"name": "Feature/pricing task"}
    memo_task = {"name": "Weekly memo task"}

print("4 tasks created with flow: track -> launch/feature analysis -> weekly memo")


4 tasks created with flow: track -> launch/feature analysis -> weekly memo


## Step 5 — Run the Crew

In [6]:
class _DemoTaskOutput:
    def __init__(self, name, raw):
        self.name = name
        self.raw = raw


class _DemoResult:
    def __init__(self):
        self.raw = "Weekly memo generated: competitor price shifts accelerating, launch velocity rising, and two priority responses recommended."
        self.tasks_output = [
            _DemoTaskOutput("Tracking", "Activities categorized by threat and momentum signal."),
            _DemoTaskOutput("Launch Analysis", "Launch impact and urgency assessed per competitor update."),
            _DemoTaskOutput("Feature/Pricing Analysis", "Comparison matrix updated with win/loss markers."),
            _DemoTaskOutput("Weekly Memo", "Final weekly memo drafted for leadership and GTM teams."),
        ]


if CREWAI_AVAILABLE:
    intel_crew = Crew(
        agents=[tracker, launch_summarizer, feature_pricing_analyst, memo_writer],
        tasks=[track_task, launch_task, feature_task, memo_task],
        process=Process.sequential,
        verbose=True,
    )

    print("Competitive intelligence crew assembled.")
    result = intel_crew.kickoff()
    print("\nWeekly memo generated.")
else:
    print("Demo mode: showing specialized intelligence flow without live CrewAI kickoff.")
    result = _DemoResult()


Demo mode: showing specialized intelligence flow without live CrewAI kickoff.


In [7]:
def preview(label, text):
    print("\n" + ("=" * 60))
    print(label)
    print("=" * 60)
    print(text)


print("WEEKLY COMPETITIVE INTELLIGENCE MEMO")
print("=" * 60)
print(result.raw)

preview("Tracking Output", result.tasks_output[0].raw)
preview("Launch Output", result.tasks_output[1].raw)
preview("Feature/Pricing Output", result.tasks_output[2].raw)
preview("Weekly Memo Output", result.tasks_output[3].raw)


WEEKLY COMPETITIVE INTELLIGENCE MEMO
Weekly memo generated: competitor price shifts accelerating, launch velocity rising, and two priority responses recommended.

Tracking Output
Activities categorized by threat and momentum signal.

Launch Output
Launch impact and urgency assessed per competitor update.

Feature/Pricing Output
Comparison matrix updated with win/loss markers.

Weekly Memo Output
Final weekly memo drafted for leadership and GTM teams.


## Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| Agent specialization | Each agent owns one analysis layer |
| Evidence flow | Tracking feeds launch and pricing/feature analysis before memo synthesis |
| Weekly decision output | Single memo optimized for fast leadership readout |
| Structured execution | Sequential orchestration with explicit context handoffs |

## Extensions

- Automate source ingestion from competitor blogs and changelogs
- Add historical trend scoring across weekly memos
- Export memo to Slack and email digests
- Add confidence scores per claim in memo sections
